# Multi-product validation of SWOT water surface elevation in the Pantanal Rivers

This single-notebook workflow is linked to a manuscript currently under submission and not yet published:

**Multi-product validation of SWOT water surface elevation in the Pantanal Rivers**

Jahdy Moreno-Oliveira a,b*, Alice Fassoni-Andrade b, Mark A. Trigg c, Daniel Medeiros Moreira d, Evlyn Marcia Leão de Moraes Novoa

a National Institute for Space Research (INPE), São Jose dos Campos, Brazil  
b Institute of Geosciences, University of Brasília (UnB), Brasília, Brazil  
c School of Civil Engineering, University of Leeds, Leeds, United Kingdom  
d Geological Survey of Brazil, Rio de Janeiro, Brazil

Manuscript status: under submission; not yet published.

This notebook is intentionally self-contained. It does not require a companion `.py` file. It uses only Python standard-library modules for reading tables, matching time series, filtering outliers, and calculating metrics. If `matplotlib` is already available, it also exports figures; if not, table outputs are still generated.


In [ ]:
# Optional: uncomment only if your environment does not already have plotting support.
# The notebook generates the CSV results without installing extra packages.
# %pip install matplotlib


## Input Contract

Organize the analysis folder into a few logical data groups. The folder and file names below are generic examples; if your local names differ, update the first configuration cell so each key points to the correct location. The important rule is that the same `station_id` must be used consistently across metadata, in situ files, SWOT product tables, and altimetry tables.

```text
<analysis-root>/
├── metadata/
│   ├── station_metadata.csv
│   └── altimetry_station_codes.csv        # optional, only needed before external-data download
├── in_situ/
│   ├── raw_or_unleveled/
│   │   └── {station_id}.csv
│   └── gnss_leveled/
│       └── {station_id}_leveled.csv
├── satellite_timeseries/
│   └── {station_id}/
│       ├── swot_pixc_timeseries.csv
│       ├── swot_raster_timeseries.csv
│       ├── swot_node_timeseries.csv
│       ├── swot_reach_timeseries.csv
│       └── external_altimetry_timeseries.csv
└── results/
    ├── absolute_validation/
    ├── dynamic_validation/
    ├── driver_analysis/
    └── figures/
```

Required file types and content:

- Station metadata: one CSV table with one row per gauge/station. It must include a station identifier column such as `station`, `station_id`, `gauge_id`, `codigo`, `cod_ana`, `codigo_ana`, or `id`. Optional but useful columns include station name, river, latitude, longitude, river width, drainage area, Pantanal subregion, and land-cover percentages.
- In situ water-level series: one CSV per station. Each file must include a parseable date/time column and one water-level column in meters. Accepted water-level aliases include `wse`, `wse_ana`, `wse_egm08`, `Cota`, `cota`, `nivel`, or `stage`. GNSS-leveled files should use the same station identifier as the raw/unleveled files.
- SWOT product time series: one prepared CSV per product and station. Each file must include a parseable acquisition date/time column and a WSE column in meters. PIXC and Raster tables should provide a mean WSE column such as `wse_media` or `wse`; Node and Reach tables should provide `wse` or `WSE`. Product quality columns are optional but used when available.
- External altimetry series: one CSV per station or one station-specific folder containing the consolidated series. Each row must include a parseable date/time column and a WSE column in meters, usually named `wse`.
- Output folders: they can be empty before execution. The notebook creates CSV tables, diagnostics, and figures there.



Column and date formats:

- Preferred date/time format: ISO `YYYY-MM-DD` or `YYYY-MM-DD HH:MM:SS`. SWOT filename/date columns may also contain a `YYYYMMDDTHHMMSS` token, for example `SWOT_L2_20240415T123456_...nc`. The notebook also accepts `YYYY/MM/DD`, `YYYYMMDD`, `DD/MM/YYYY`, `DD-MM-YYYY`, Unix timestamps, and Excel serial dates when they are unambiguous. Avoid ambiguous `MM/DD/YYYY` dates.
- Date/time values are treated as timezone-naive after parsing. Use one time basis consistently across all input tables.
- Water-surface elevation values must be numeric and in meters. Decimal points are preferred; decimal commas are accepted when the rest of the value is numeric.
- Missing numeric values should be blank or standard missing-value tokens such as `NaN`, `NA`, or `null`. Do not use text such as `no data` inside numeric columns.
- Station identifiers should be stored as text or integer-like values without changing leading/significant digits. The same identifier must appear in file names, metadata, and station folders.

Expected columns by CSV type:

| CSV type | Required date column names | Required WSE column names | Optional columns used by the notebook | Notes |
|---|---|---|---|---|
| Station metadata | not required | not required | `station_name`, `name`, `nome`, `river`, `rio`, `latitude`, `longitude`, `width`, `drainage_area`, `subregion`, `pantanal` | Must include a station identifier column: `station`, `station_id`, `gauge_id`, `codigo`, `cod_ana`, `codigo_ana`, or `id`. |
| In situ raw/unleveled | `data`, `date`, `datetime`, `data_ref`, or `time` | `wse_ana`, `Cota`, `cota`, `wse`, `WSE`, `wse_egm08`, `nivel`, or `stage` | `hydro_phase`, `phase`, `fase`, `fase_hidrologica`, `HydroPhase` | Used for dynamic validation when the station is not GNSS-leveled, and as fallback when a leveled file is unavailable. |
| In situ GNSS-leveled | `data`, `date`, `datetime`, `data_ref`, or `time` | `wse_swot_ref`, `wse_ana`, `wse_egm08`, `Cota`, `cota`, `wse`, `WSE`, `nivel`, or `stage` | `hydro_phase`, `phase`, `fase`, `fase_hidrologica`, `HydroPhase` | Used for absolute validation. Values must be in the vertical reference used for comparison with satellite WSE. |
| SWOT PIXC | `data`, `date`, `datetime`, `arquivo`, or `filename` | `wse_media`, `wse`, `WSE`, `height_media`, `height`, or `height_mean` | source filename columns such as `arquivo`; quality/provenance columns | If date is stored in a filename column, include a `YYYYMMDDTHHMMSS` token. |
| SWOT Raster | `data`, `date`, `datetime`, `arquivo`, or `filename` | `wse_media`, `wse`, `WSE`, `height`, or `height_mean` | `wse_qual_media`; source filename columns | Rows with `wse_qual_media >= 1.8` are removed when that column is present. |
| SWOT Node | `data`, `date`, `datetime`, `time_str`, `timestamp_arquivo`, `arquivo_origem`, or `arquivo` | `wse`, `WSE`, or `wse_sm` | `node_q`; source filename columns | Node quality filtering follows the configured `node_q` threshold/fallback rule when `node_q` is present. |
| SWOT Reach | `data`, `date`, `datetime`, `time_str`, `timestamp_arquivo`, `arquivo_origem`, or `arquivo` | `wse` or `WSE` | `reach_q`; source filename columns | Reach quality filtering follows the configured `reach_q` threshold/fallback rule when `reach_q` is present. |
| External altimetry | `date`, `data`, `datetime`, or `time` | `wse`, `WSE`, `water_level`, `height`, or `altura` | none required | Multiple rows on the same date are averaged before matchup. |

Minimal CSV header examples:

```text
station_metadata.csv: station,station_name,river,latitude,longitude,width,drainage_area
in_situ/{station_id}.csv: data,wse
in_situ_leveled/{station_id}_leveled.csv: data,wse_swot_ref
swot_pixc_timeseries.csv: data,wse_media
swot_raster_timeseries.csv: data,wse_media,wse_qual_media
swot_node_timeseries.csv: time_str,wse,node_q
swot_reach_timeseries.csv: time_str,wse,reach_q
external_altimetry_timeseries.csv: date,wse
```

Before running the notebook, check the first configuration cell and map each logical input group above to the actual folder names used in your environment.


In [ ]:
from pathlib import Path

def detect_base_path():
    cwd = Path.cwd()
    candidates = [cwd, cwd / "scripts"]
    if cwd.name == "scripts":
        candidates.insert(0, cwd)
    markers = ["analise_final", "dados_processados", "analises"]
    for candidate in candidates:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return cwd

BASE_PATH = detect_base_path()

CFG = {
    "base_path": BASE_PATH,
    "date_start": "2023-03-31",
    "date_end": "2025-08-01",
    "calval_cutoff_date": "2023-07-01",
    "match_window_days": 3,
    "station_filter": [],
    "stop_if_no_matchups": True,
    "node_reach_qc_mode": "fallback",
    "node_reach_qc_min_rows_for_fallback": 3,
    "metadata_candidates": [
        Path("analises/tabelas/ana_metadados.csv"),
        Path("ana_metadados.csv"),
    ],
    "in_situ_unleveled_dir": Path("dados_processados/ana/sem_nivelamento"),
    "in_situ_gnss_leveled_dir": Path("dados_processados/ana/nivelamento"),
    "station_dir_template": "analise_final/Estacao_{station}",
    "absolute_dir": Path("analysis_absolute"),
    "temporal_dir": Path("analysis_temporal"),
    "drivers_dir": Path("analysis_drivers"),
    "figures_main_dir": Path("figures/main"),
    "figures_supplementary_dir": Path("figures/supplementary"),
}

CFG


In [ ]:

import csv
import json
import math
import re
import unicodedata
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path
from statistics import median

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

PRODUCTS_SWOT = ["pixc", "raster", "node", "reach"]
PRODUCTS_ALL = PRODUCTS_SWOT + ["altimetry"]
PRODUCT_LABELS = {"pixc": "PIXC", "raster": "Raster", "node": "Node", "reach": "Reach", "altimetry": "Altimetry"}
PRODUCT_COLORS = {"pixc": "#e67e22", "raster": "#2e7d32", "node": "#c0392b", "reach": "#6c5ce7", "altimetry": "#6f6f6f"}

SWOT_SCHEMAS = {
    "pixc": {"file": "pixc_complete_raw.csv", "date_cols": ["data", "date", "datetime", "arquivo", "filename"], "wse_cols": ["wse_media", "wse", "WSE", "height_media", "height", "height_mean"]},
    "raster": {"file": "raster_complete_raw.csv", "date_cols": ["data", "date", "datetime", "arquivo", "filename"], "wse_cols": ["wse_media", "wse", "WSE", "height", "height_mean"]},
    "node": {"file": "node_complete_raw.csv", "date_cols": ["data", "date", "datetime", "time_str", "timestamp_arquivo", "arquivo_origem", "arquivo"], "wse_cols": ["wse", "WSE", "wse_sm"]},
    "reach": {"file": "reach_complete_raw.csv", "date_cols": ["data", "date", "datetime", "time_str", "timestamp_arquivo", "arquivo_origem", "arquivo"], "wse_cols": ["wse", "WSE"]},
}

MATCHUP_FIELDS = [
    "station", "station_name", "river", "product", "station_group", "lineage_stage",
    "date_ref", "date_sat", "days_delta", "wse_insitu", "wse_sat", "signed_error", "error", "abs_error",
    "hydro_phase", "orbital_phase", "wse_source_column", "time_source_column", "source_file",
]
METRIC_FIELDS = ["station", "product", "n_obs", "mae", "rmse", "spearman_r", "bias", "n_stations"]
LOAD_AUDIT_FIELDS = [
    "station", "source", "product", "station_group", "file_path", "exists", "rows_file", "rows_loaded",
    "rows_after_date_filter", "date_source_column", "wse_source_column", "reason",
]
OUTLIER_AUDIT_FIELDS = ["station", "product", "station_group", "rows_before", "rows_after_quality_filter", "rows_after_outlier_filter", "rows_removed", "removal_reason"]

DATA_LOAD_AUDIT = []


def pjoin(*parts):
    first = Path(parts[0]) if parts else Path(CFG["base_path"])
    if first.is_absolute():
        return first.joinpath(*parts[1:])
    return Path(CFG["base_path"]).joinpath(*parts)


def cfg_path(key):
    return pjoin(CFG[key])


def ensure_dirs():
    for key in ["absolute_dir", "temporal_dir", "drivers_dir", "figures_main_dir", "figures_supplementary_dir"]:
        cfg_path(key).mkdir(parents=True, exist_ok=True)


def canonical(value):
    text = unicodedata.normalize("NFKD", str(value or ""))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text.strip().lower()).strip("_")
    return text


def read_text_sample(path, size=8192):
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        return f.read(size)


def read_table(path):
    path = Path(path)
    if not path.exists():
        return []
    sample = read_text_sample(path)
    delimiters = [",", ";", "\t", "|"]
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters="".join(delimiters))
        delimiters.insert(0, dialect.delimiter)
    except Exception:
        pass
    attempts = []
    for delimiter in dict.fromkeys(delimiters):
        try:
            with open(path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
                reader = csv.DictReader(f, delimiter=delimiter)
                rows = [{(k or "").strip(): (v or "").strip() for k, v in row.items()} for row in reader]
                attempts.append(rows)
        except Exception:
            continue
    if not attempts:
        return []
    return max(attempts, key=lambda rows: (len(rows[0]) if rows else 0, len(rows)))


def write_table(path, rows, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = rows or []
    if fieldnames is None:
        keys = []
        seen = set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    keys.append(key)
        fieldnames = keys or ["status"]
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})


def normalize_station_id(value):
    text = str(value or "").replace("\ufeff", "").replace("\xa0", " ").strip()
    text = re.sub(r"\s+", "", text)
    if re.fullmatch(r"\d+\.0+", text):
        return text.split(".")[0]
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except Exception:
        pass
    return text


def to_float(value):
    if value is None:
        return None
    text = str(value).strip().replace("%", "").replace(" ", "")
    if text == "" or text.lower() in {"nan", "none", "null", "na", "nat"}:
        return None
    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "").replace(",", ".")
        else:
            text = text.replace(",", "")
    elif "," in text:
        text = text.replace(",", ".")
    text = re.sub(r"[^0-9eE+\-.]", "", text)
    try:
        out = float(text)
        return out if math.isfinite(out) else None
    except Exception:
        return None


def parse_date(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    token = re.search(r"(\d{8}T\d{6})", text)
    if token:
        text = token.group(1)
    else:
        token = re.search(r"(\d{4}[-_/]?\d{2}[-_/]?\d{2})", text)
        if token and not re.search(r"\d{1,2}/\d{1,2}/\d{4}", text):
            text = token.group(1)
    text_iso = text.replace("Z", "+00:00")
    try:
        dt = datetime.fromisoformat(text_iso)
        return dt.replace(tzinfo=None)
    except Exception:
        pass
    patterns = [
        "%Y-%m-%d", "%Y/%m/%d", "%Y%m%d", "%d/%m/%Y", "%d-%m-%Y",
        "%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M:%S", "%Y%m%dT%H%M%S",
    ]
    for pattern in patterns:
        try:
            width = len(datetime.now().strftime(pattern))
            return datetime.strptime(text[:width], pattern)
        except Exception:
            pass
    try:
        number = float(text)
        if 10000 < number < 80000:
            return datetime(1899, 12, 30) + timedelta(days=number)
        if number > 1e12:
            return datetime.fromtimestamp(number / 1e9)
        if number > 1e9:
            return datetime.fromtimestamp(number)
    except Exception:
        pass
    return None


def date_key(dt):
    return dt.strftime("%Y-%m-%d") if isinstance(dt, datetime) else ""


def between_dates(dt):
    if not dt:
        return False
    return datetime.fromisoformat(CFG["date_start"]) <= dt <= datetime.fromisoformat(CFG["date_end"])


def orbital_phase(dt):
    if not dt:
        return ""
    return "CalVal" if dt < datetime.fromisoformat(CFG["calval_cutoff_date"]) else "Nominal"


def pick(row, candidates):
    row_map = {canonical(k): k for k in row.keys()}
    for candidate in candidates:
        key = row_map.get(canonical(candidate))
        if key is not None and str(row.get(key, "")).strip() != "":
            return row[key], key
    return "", ""


def choose_numeric_column(rows, preferred, exclude=()):
    if not rows:
        return None, []
    columns = list(rows[0].keys())
    ordered = []
    for name in preferred:
        for col in columns:
            if canonical(col) == canonical(name) and col not in ordered:
                ordered.append(col)
    for col in columns:
        c = canonical(col)
        if col not in ordered and any(token in c for token in ["wse", "height", "cota", "nivel", "stage"]):
            ordered.append(col)
    exclude_keys = {canonical(x) for x in exclude}
    best_col = None
    best_vals = []
    best_count = -1
    for col in ordered:
        if canonical(col) in exclude_keys or any(token in canonical(col) for token in ["qual", "flag", "count", "n_pix", "uncert"]):
            continue
        vals = [to_float(row.get(col)) for row in rows]
        count = sum(v is not None and v > -1e9 for v in vals)
        if count > best_count:
            best_col = col
            best_vals = vals
            best_count = count
    return best_col, best_vals


def ranks(values):
    indexed = sorted((value, idx) for idx, value in enumerate(values))
    out = [0.0] * len(values)
    i = 0
    while i < len(indexed):
        j = i
        while j + 1 < len(indexed) and indexed[j + 1][0] == indexed[i][0]:
            j += 1
        rank = (i + j + 2) / 2.0
        for _, idx in indexed[i:j + 1]:
            out[idx] = rank
        i = j + 1
    return out


def pearson(x, y):
    n = len(x)
    if n < 2:
        return None
    mx = sum(x) / n
    my = sum(y) / n
    sx = sum((v - mx) ** 2 for v in x)
    sy = sum((v - my) ** 2 for v in y)
    if sx == 0 or sy == 0:
        return None
    return sum((a - mx) * (b - my) for a, b in zip(x, y)) / math.sqrt(sx * sy)


def spearman(x, y):
    clean = [(a, b) for a, b in zip(x, y) if a is not None and b is not None]
    if len(clean) < 2:
        return None
    return pearson(ranks([a for a, _ in clean]), ranks([b for _, b in clean]))


def rmse(errors):
    return math.sqrt(sum(e * e for e in errors) / len(errors)) if errors else None


def infer_hydro_phase(values):
    clean = [v for v in values if v is not None]
    if len(clean) < 5:
        return ["" for _ in values]
    ordered = sorted(clean)
    q25 = ordered[int(0.25 * (len(ordered) - 1))]
    q50 = ordered[int(0.50 * (len(ordered) - 1))]
    q75 = ordered[int(0.75 * (len(ordered) - 1))]
    out = []
    previous = None
    for value in values:
        if value is None:
            out.append("")
        elif value <= q25:
            out.append("Low")
        elif value > q75:
            out.append("High")
        elif previous is None:
            out.append("Rising")
        else:
            out.append("Rising" if value >= previous else "Falling")
        previous = value if value is not None else previous
    return out


def metadata_candidates():
    return [pjoin(p) for p in CFG["metadata_candidates"]]


def station_dirs():
    root = pjoin(Path(CFG["station_dir_template"].split("/Estacao_")[0]))
    dirs = sorted(root.glob("Estacao_*")) if root.exists() else []
    if not dirs:
        dirs = sorted(pjoin(".").glob("**/Estacao_*"))
    return [d for d in dirs if d.is_dir() and ".ipynb_checkpoints" not in str(d)]


def load_metadata():
    path = next((candidate for candidate in metadata_candidates() if candidate.exists()), None)
    if path is None:
        checked = ", ".join(str(p) for p in metadata_candidates())
        raise FileNotFoundError(f"Station metadata not found. Checked: {checked}")
    rows = read_table(path)
    out = []
    seen = set()
    for row in rows:
        station, _ = pick(row, ["station", "station_id", "gauge_id", "codigo", "cod_ana", "codigo_ana", "id"])
        station = normalize_station_id(station)
        if not station or station in seen:
            continue
        if CFG["station_filter"] and station not in set(map(str, CFG["station_filter"])):
            continue
        out.append({
            "station": station,
            "station_name": pick(row, ["station_name", "nome", "name"])[0],
            "river": pick(row, ["river", "rio", "river_name"])[0],
            "latitude": pick(row, ["latitude", "lat"])[0],
            "longitude": pick(row, ["longitude", "lon", "long"])[0],
            "width": pick(row, ["width", "largura"])[0],
            "drainage_area": pick(row, ["drainage_area", "area_drenagem"])[0],
            "subregion": pick(row, ["subregion", "subregiao"])[0],
            "pantanal": pick(row, ["pantanal"])[0],
        })
        seen.add(station)
    return out


def add_load_audit(**kwargs):
    DATA_LOAD_AUDIT.append({key: kwargs.get(key, "") for key in LOAD_AUDIT_FIELDS})


def load_insitu(station, leveled=False):
    base = cfg_path("in_situ_gnss_leveled_dir" if leveled else "in_situ_unleveled_dir")
    station_group = "leveled" if leveled else "unleveled"
    if leveled:
        candidates = [base / f"{station}_nivelada.csv", base / f"{station}.csv", base / f"{station}_leveled.csv"]
        candidates.extend(sorted(base.glob(f"{station}*nivel*.csv")) if base.exists() else [])
    else:
        candidates = [base / f"{station}.csv"]
        candidates.extend(sorted(base.glob(f"{station}*.csv")) if base.exists() else [])
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        add_load_audit(station=station, source="in_situ", station_group=station_group, file_path=str(candidates[0]), exists=False, reason="file_not_found")
        return []
    raw = read_table(path)
    if not raw:
        add_load_audit(station=station, source="in_situ", station_group=station_group, file_path=str(path), exists=True, rows_file=0, reason="empty_file")
        return []
    date_values = []
    date_col = ""
    for row in raw:
        value, col = pick(row, ["data", "date", "datetime", "data_ref", "time"])
        date_values.append(parse_date(value))
        date_col = date_col or col
    wse_col, wse_values = choose_numeric_column(raw, ["wse_swot_ref", "wse_ana", "wse_egm08", "Cota", "cota", "wse", "WSE", "nivel", "stage"])
    out = []
    hydro_values = []
    for row, dt, wse in zip(raw, date_values, wse_values):
        if not dt or wse is None:
            continue
        if not between_dates(dt):
            continue
        phase, _ = pick(row, ["hydro_phase", "phase", "fase", "fase_hidrologica", "HydroPhase"])
        out.append({"station": station, "data": dt, "date": date_key(dt), "wse_insitu": wse, "hydro_phase": phase})
        hydro_values.append(wse)
    inferred = infer_hydro_phase(hydro_values)
    for row, phase in zip(out, inferred):
        if not row["hydro_phase"]:
            row["hydro_phase"] = phase
    add_load_audit(
        station=station, source="in_situ", station_group=station_group, file_path=str(path), exists=True,
        rows_file=len(raw), rows_loaded=len(out), rows_after_date_filter=len(out), date_source_column=date_col,
        wse_source_column=wse_col or "", reason="ok" if out else "no_rows_after_date_or_wse_filter",
    )
    return sorted(out, key=lambda r: r["data"])


def station_dir(station):
    direct = pjoin(CFG["station_dir_template"].format(station=station))
    if direct.exists():
        return direct
    matches = sorted(pjoin(".").glob(f"**/Estacao_{station}"))
    return matches[0] if matches else direct


def load_swot(station, product):
    schema = SWOT_SCHEMAS[product]
    path = station_dir(station) / schema["file"]
    if not path.exists():
        recursive = sorted(pjoin(".").glob(f"**/Estacao_{station}/{schema['file']}"))
        recursive = [p for p in recursive if ".ipynb_checkpoints" not in str(p)]
        path = recursive[0] if recursive else path
    if not path.exists():
        add_load_audit(station=station, source="swot", product=product, file_path=str(path), exists=False, reason="file_not_found")
        return []
    raw = read_table(path)
    if not raw:
        add_load_audit(station=station, source="swot", product=product, file_path=str(path), exists=True, rows_file=0, reason="empty_file")
        return []
    date_col = ""
    parsed_dates = []
    for row in raw:
        value, col = pick(row, schema["date_cols"])
        parsed_dates.append(parse_date(value))
        date_col = date_col or col
    wse_col, wse_values = choose_numeric_column(raw, schema["wse_cols"])
    rows = []
    for row, dt, wse in zip(raw, parsed_dates, wse_values):
        if not dt or wse is None or not between_dates(dt) or wse <= -1e9:
            continue
        out = dict(row)
        out.update({
            "station": station, "product": product, "data": dt, "date": date_key(dt), "wse_sat": wse,
            "orbital_phase": orbital_phase(dt), "wse_source_column": wse_col or "", "time_source_column": date_col or "",
            "source_file": str(path),
        })
        rows.append(out)
    add_load_audit(
        station=station, source="swot", product=product, file_path=str(path), exists=True,
        rows_file=len(raw), rows_loaded=len(rows), rows_after_date_filter=len(rows), date_source_column=date_col,
        wse_source_column=wse_col or "", reason="ok" if rows else "no_rows_after_date_or_wse_filter",
    )
    return sorted(rows, key=lambda r: r["data"])


def load_altimetry(station):
    base = station_dir(station)
    files = sorted(base.glob("alt_conv_*.csv")) if base.exists() else []
    if not files:
        files = sorted(pjoin(".").glob(f"**/Estacao_{station}/alt_conv_*.csv"))
        files = [p for p in files if ".ipynb_checkpoints" not in str(p)]
    if not files:
        add_load_audit(station=station, source="altimetry", product="altimetry", file_path=str(base / "alt_conv_*.csv"), exists=False, reason="file_not_found")
        return []
    grouped = defaultdict(list)
    rows_file = 0
    used_col = ""
    used_date = ""
    for path in files:
        raw = read_table(path)
        rows_file += len(raw)
        if not raw:
            continue
        date_col = ""
        parsed_dates = []
        for row in raw:
            value, col = pick(row, ["date", "data", "datetime", "time"])
            parsed_dates.append(parse_date(value))
            date_col = date_col or col
        wse_col, wse_values = choose_numeric_column(raw, ["wse", "WSE", "water_level", "height", "altura"])
        used_col = used_col or (wse_col or "")
        used_date = used_date or date_col
        for dt, wse in zip(parsed_dates, wse_values):
            if dt and wse is not None and between_dates(dt):
                grouped[date_key(dt)].append(wse)
    out = []
    for d, values in grouped.items():
        dt = datetime.fromisoformat(d)
        out.append({"station": station, "product": "altimetry", "data": dt, "date": d, "wse_sat": sum(values) / len(values), "orbital_phase": orbital_phase(dt), "wse_source_column": used_col, "time_source_column": used_date, "source_file": ";".join(str(f) for f in files)})
    add_load_audit(
        station=station, source="altimetry", product="altimetry", file_path=";".join(str(f) for f in files), exists=True,
        rows_file=rows_file, rows_loaded=len(out), rows_after_date_filter=len(out), date_source_column=used_date,
        wse_source_column=used_col, reason="ok" if out else "no_rows_after_date_or_wse_filter",
    )
    return sorted(out, key=lambda r: r["data"])


def quality_filter(rows, product):
    quality = [dict(row) for row in rows]
    if product in PRODUCTS_SWOT:
        filtered = []
        for row in quality:
            provenance = " ".join(str(row.get(k, "")) for k in ["arquivo", "arquivo_origem", "source_file"]).upper()
            if any(tag in provenance for tag in ["PIC0", "PGC0"]):
                continue
            filtered.append(row)
        quality = filtered

    if product in {"node", "reach"}:
        q_col = "node_q" if product == "node" else "reach_q"
        if any(to_float(row.get(q_col)) is not None for row in quality):
            strict = [row for row in quality if to_float(row.get(q_col)) is not None and to_float(row.get(q_col)) < 2.0]
            if CFG.get("node_reach_qc_mode", "fallback") == "strict_only":
                quality = strict
            elif len(strict) >= int(CFG.get("node_reach_qc_min_rows_for_fallback", 3)):
                quality = strict
            else:
                quality = [row for row in quality if to_float(row.get(q_col)) is not None and to_float(row.get(q_col)) < 3.0]
    elif product == "raster" and any(to_float(row.get("wse_qual_media")) is not None for row in quality):
        quality = [row for row in quality if to_float(row.get("wse_qual_media")) is not None and to_float(row.get("wse_qual_media")) < 1.8]
    return quality

def hampel_filter(rows, window=5, n_sigma=3.0):
    if not rows:
        return [], {"rows_before": 0, "rows_after_quality_filter": 0, "rows_after_outlier_filter": 0, "rows_removed": 0, "removal_reason": "no_data"}
    values = [to_float(r.get("wse_sat")) for r in rows]
    keep = [True] * len(rows)
    half = max(1, window // 2)
    for i, value in enumerate(values):
        if value is None:
            keep[i] = False
            continue
        lo = max(0, i - half)
        hi = min(len(values), i + half + 1)
        local = [v for v in values[lo:hi] if v is not None]
        if len(local) < 3:
            continue
        med = median(local)
        mad = median([abs(v - med) for v in local])
        if mad == 0:
            continue
        if abs(value - med) > n_sigma * 1.4826 * mad:
            keep[i] = False
    filtered = [dict(r) for r, flag in zip(rows, keep) if flag]
    return filtered, {
        "rows_before": len(rows),
        "rows_after_quality_filter": len(rows),
        "rows_after_outlier_filter": len(filtered),
        "rows_removed": len(rows) - len(filtered),
        "removal_reason": "hampel_wse_sat" if len(rows) != len(filtered) else "none",
    }


def nearest_match_no_reuse(ref_rows, sat_rows):
    ref_rows = sorted(ref_rows, key=lambda r: r["data"])
    sat_rows = sorted(sat_rows, key=lambda r: r["data"])
    used = set()
    window = timedelta(days=CFG["match_window_days"])
    matched = []
    for ref in ref_rows:
        candidates = []
        for idx, sat in enumerate(sat_rows):
            if idx in used:
                continue
            delta = abs(sat["data"] - ref["data"])
            if delta <= window:
                candidates.append((delta, idx, sat))
        if not candidates:
            continue
        delta, idx, sat = min(candidates, key=lambda item: item[0])
        used.add(idx)
        row = dict(sat)
        row.update({
            "date_ref": date_key(ref["data"]),
            "date_sat": date_key(sat["data"]),
            "days_delta": round(delta.total_seconds() / 86400, 3),
            "wse_insitu": ref["wse_insitu"],
            "wse_sat": sat["wse_sat"],
            "hydro_phase": ref.get("hydro_phase", ""),
            "orbital_phase": sat.get("orbital_phase", ""),
        })
        row.pop("data", None)
        matched.append(row)
    return matched


def available_leveled_stations():
    base = cfg_path("in_situ_gnss_leveled_dir")
    stations = set()
    if base.exists():
        for path in base.glob("*.csv"):
            stem = path.stem
            for suffix in ["_nivelada", "_leveled"]:
                if stem.endswith(suffix):
                    stem = stem[: -len(suffix)]
            stations.add(normalize_station_id(stem))
    stations.discard("")
    return stations


def add_matches(stages, audit, meta, station, product, ref_rows, sat_stage_tables, station_group):
    for stage, rows in sat_stage_tables.items():
        matches = nearest_match_no_reuse(ref_rows, rows)
        for match in matches:
            error = to_float(match["wse_sat"]) - to_float(match["wse_insitu"])
            match.update(meta)
            match.update({
                "station": station, "product": product, "station_group": station_group, "lineage_stage": stage,
                "signed_error": error, "error": error, "abs_error": abs(error),
            })
        stages[stage].extend(matches)


def satellite_stage_tables(station, product, station_group):
    sat = load_altimetry(station) if product == "altimetry" else load_swot(station, product)
    quality = quality_filter(sat, product)
    outlier, audit_row = hampel_filter(quality)
    audit_row.update({"station": station, "product": product, "station_group": station_group, "rows_after_quality_filter": len(quality)})
    return {"raw": sat, "quality_filtered": quality, "outlier_filtered": outlier}, audit_row


def load_reference_for_station(station, is_leveled):
    primary = load_insitu(station, leveled=is_leveled)
    if primary:
        return primary
    return load_insitu(station, leveled=not is_leveled)


def build_matchups(metadata):
    DATA_LOAD_AUDIT.clear()
    leveled = available_leveled_stations()
    stages = {"raw": [], "quality_filtered": [], "outlier_filtered": []}
    audit = []
    for meta in metadata:
        station = meta["station"]
        is_leveled = station in leveled
        station_group = "leveled" if is_leveled else "nonleveled"
        ref_rows = load_reference_for_station(station, is_leveled)
        if not ref_rows:
            continue
        for product in PRODUCTS_ALL:
            sat_tables, audit_row = satellite_stage_tables(station, product, station_group)
            audit.append(audit_row)
            add_matches(stages, audit, meta, station, product, ref_rows, sat_tables, station_group)
    return stages, audit

def group_by(rows, keys):
    groups = defaultdict(list)
    for row in rows:
        groups[tuple(row.get(k, "") for k in keys)].append(row)
    return groups


def metric_values(group):
    clean = []
    for row in group:
        ref = to_float(row.get("wse_insitu"))
        sat = to_float(row.get("wse_sat"))
        if ref is None or sat is None:
            continue
        clean.append((ref, sat, sat - ref, row))
    return clean


def aggregate_metrics(rows, keys):
    out = []
    for values, group in group_by(rows, keys).items():
        rec = {k: v for k, v in zip(keys, values)}
        clean = metric_values(group)
        errors = [e for _, _, e, _ in clean]
        ref = [r for r, _, _, _ in clean]
        sat = [s for _, s, _, _ in clean]
        pooled_rho = spearman(ref, sat)

        station_rhos = []
        for (_,), station_group in group_by(group, ["station"]).items():
            station_clean = metric_values(station_group)
            if len(station_clean) < 2:
                continue
            station_ref = [r for r, _, _, _ in station_clean]
            station_sat = [s for _, s, _, _ in station_clean]
            rho = spearman(station_ref, station_sat)
            if rho is not None:
                station_rhos.append(rho)

        station_mean = sum(station_rhos) / len(station_rhos) if station_rhos else None
        station_median = median(station_rhos) if station_rhos else None
        rec.update({
            "n_obs": len(errors),
            "mae": sum(abs(e) for e in errors) / len(errors) if errors else "",
            "rmse": rmse(errors) if errors else "",
            "spearman_r_pooled": pooled_rho if pooled_rho is not None else "",
            "spearman_p_pooled": "",
            "n_station_spearman": len(station_rhos),
            "spearman_r_station_mean": station_mean if station_mean is not None else "",
            "spearman_r_station_median": station_median if station_median is not None else "",
            "spearman_r": station_mean if station_mean is not None else (pooled_rho if pooled_rho is not None else ""),
            "bias": sum(errors) / len(errors) if errors else "",
            "n_stations": len(set(r.get("station", "") for r in group)),
        })
        out.append(rec)
    return sorted(out, key=lambda r: tuple(str(r.get(k, "")) for k in keys))

def dynamic_metrics(rows):
    station_rows = []
    normalized = []
    for (station, product), group in group_by(rows, ["station", "product"]).items():
        group = sorted(group, key=lambda r: r.get("date_ref", ""))
        if len(group) < 5:
            continue
        ref_vals = [to_float(r.get("wse_insitu")) for r in group]
        sat_vals = [to_float(r.get("wse_sat")) for r in group]
        valid = [(row, ref, sat) for row, ref, sat in zip(group, ref_vals, sat_vals) if ref is not None and sat is not None]
        if len(valid) < 5:
            continue
        group = [row for row, _, _ in valid]
        ref_vals = [ref for _, ref, _ in valid]
        sat_vals = [sat for _, _, sat in valid]
        if max(ref_vals) == min(ref_vals) or max(sat_vals) == min(sat_vals):
            continue
        ref_norm = [(v - min(ref_vals)) / (max(ref_vals) - min(ref_vals)) for v in ref_vals]
        sat_norm = [(v - min(sat_vals)) / (max(sat_vals) - min(sat_vals)) for v in sat_vals]
        rho = spearman(ref_norm, sat_norm)
        station_rows.append({
            "station": station,
            "product": product,
            "n_obs": len(group),
            "spearman_r": rho if rho is not None else "",
            "spearman_p": "",
            "station_group": group[0].get("station_group", ""),
        })
        for row, rn, sn in zip(group, ref_norm, sat_norm):
            normalized.append({"station": station, "product": product, "date_ref": row.get("date_ref"), "insitu_norm": rn, "sat_norm": sn, "station_group": row.get("station_group", ""), "hydro_phase": row.get("hydro_phase", "")})
    overall = []
    for (product,), group in group_by(station_rows, ["product"]).items():
        rhos = [to_float(r.get("spearman_r")) for r in group if to_float(r.get("spearman_r")) is not None]
        mean_rho = sum(rhos) / len(rhos) if rhos else ""
        median_rho = median(rhos) if rhos else ""
        overall.append({
            "product": product,
            "n_stations": len(set(r.get("station", "") for r in group)),
            "n_station_series": len(group),
            "n_obs": sum(int(r.get("n_obs", 0)) for r in group),
            "spearman_r_mean": mean_rho,
            "spearman_r_median": median_rho,
            "mean_spearman": mean_rho,
            "median_spearman": median_rho,
        })
    return station_rows, overall, normalized

def savefig(name):
    if plt is None:
        return None
    path = cfg_path("figures_main_dir") / f"{name}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, bbox_inches="tight", dpi=300)
    plt.close()
    return str(path)


def make_figures(tables):
    manifest = []
    if plt is None:
        return manifest
    metrics = tables.get("metrics_absolute_overall", [])
    if metrics:
        for metric in ["mae", "rmse", "bias"]:
            plt.figure(figsize=(6, 4))
            xs = [PRODUCT_LABELS.get(r["product"], r["product"]) for r in metrics]
            ys = [to_float(r.get(metric)) or 0 for r in metrics]
            colors = [PRODUCT_COLORS.get(r["product"], "#777") for r in metrics]
            plt.bar(xs, ys, color=colors)
            plt.ylabel(metric.upper() if metric != "bias" else "Bias (m)")
            plt.grid(axis="y", alpha=0.25)
            out = savefig(f"fig_absolute_{metric}")
            if out:
                manifest.append({"figure_path": out})
    dyn = tables.get("metrics_dynamic_overall", [])
    if dyn:
        plt.figure(figsize=(7, 4))
        xs = [PRODUCT_LABELS.get(r["product"], r["product"]) for r in dyn]
        ys = [to_float(r.get("spearman_r_median", r.get("median_spearman"))) or 0 for r in dyn]
        colors = [PRODUCT_COLORS.get(r["product"], "#777") for r in dyn]
        plt.bar(xs, ys, color=colors)
        plt.ylim(-1, 1)
        plt.ylabel("Median Spearman")
        plt.grid(axis="y", alpha=0.25)
        out = savefig("fig_dynamic_spearman_summary")
        if out:
            manifest.append({"figure_path": out})
    return manifest


def split_by_station_group(stages, station_group):
    return {stage: [r for r in rows if r.get("station_group") == station_group] for stage, rows in stages.items()}


def write_stage_tables(out_dir, stages):
    for stage, rows in stages.items():
        write_table(out_dir / f"matchups_{stage}.csv", rows, MATCHUP_FIELDS)


def write_named_tables(out_dir, tables, output_names, field_map=None):
    field_map = field_map or {}
    for key, filenames in output_names.items():
        for filename in filenames:
            write_table(out_dir / filename, tables.get(key, []), field_map.get(key))


def driver_tables(metrics_by_station, metadata):
    meta_by_station = {m["station"]: m for m in metadata}
    matrix = []
    for row in metrics_by_station:
        rec = dict(row)
        rec.update({k: v for k, v in meta_by_station.get(row.get("station"), {}).items() if k not in rec})
        matrix.append(rec)
    summary = []
    candidate_vars = ["width", "drainage_area", "latitude", "longitude"]
    for product, group in group_by(matrix, ["product"]).items():
        y = [to_float(r.get("mae")) for r in group]
        for var in candidate_vars:
            x = [to_float(r.get(var)) for r in group]
            pairs = [(a, b) for a, b in zip(x, y) if a is not None and b is not None]
            if len(pairs) >= 3:
                summary.append({"product": product, "variable": var, "spearman_rho": spearman([a for a, _ in pairs], [b for _, b in pairs]), "n_stations": len(pairs)})
    return matrix, summary


def _display_available():
    try:
        from IPython.display import HTML, display
        return HTML, display
    except Exception:
        return None, None


def _format_number(value, digits=4):
    number = to_float(value)
    if number is None:
        return ""
    if abs(number - round(number)) < 1e-12:
        return str(int(round(number)))
    return f"{number:.{digits}f}"


def _round_table(rows, integer_cols=None, digits=4):
    integer_cols = set(integer_cols or [])
    out = []
    for row in rows:
        rec = {}
        for key, value in row.items():
            if key in integer_cols:
                number = to_float(value)
                rec[key] = "" if number is None else str(int(round(number)))
            elif to_float(value) is not None and not isinstance(value, str):
                rec[key] = _format_number(value, digits)
            else:
                rec[key] = value
        out.append(rec)
    return out


def _html_table(title, rows, columns=None, max_rows=20):
    HTML, display = _display_available()
    if display is None:
        print(f"\n{title}")
        for row in (rows or [])[:max_rows]:
            print(row)
        return
    rows = rows or []
    if columns is None:
        columns = []
        seen = set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    columns.append(key)
    shown = rows[:max_rows]
    def esc(value):
        text = str(value if value is not None else "")
        return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    header = "".join(f"<th>{esc(col)}</th>" for col in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{esc(row.get(col, ''))}</td>" for col in columns) + "</tr>"
        for row in shown
    )
    empty = "" if shown else f"<tr><td colspan='{max(1, len(columns))}'>No rows available</td></tr>"
    note = f"<div class='note'>Showing {len(shown)} of {len(rows)} rows</div>" if len(rows) > len(shown) else ""
    html = f"""
    <style>
      .swot-table-wrap {{ margin: 14px 0 22px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
      .swot-table-title {{ font-weight: 700; font-size: 15px; margin: 0 0 6px 0; color: #1f2933; }}
      .swot-table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
      .swot-table th {{ background: #eef2f6; color: #1f2933; text-align: left; padding: 8px 10px; border-bottom: 1px solid #c8d0d9; }}
      .swot-table td {{ padding: 7px 10px; border-bottom: 1px solid #e4e8ee; }}
      .swot-table tr:nth-child(even) td {{ background: #fafbfc; }}
      .swot-table .note, .note {{ color: #687385; font-size: 12px; margin-top: 5px; }}
    </style>
    <div class='swot-table-wrap'>
      <div class='swot-table-title'>{esc(title)}</div>
      <table class='swot-table'><thead><tr>{header}</tr></thead><tbody>{body or empty}</tbody></table>
      {note}
    </div>
    """
    display(HTML(html))


def product_label(product):
    return PRODUCT_LABELS.get(str(product).lower(), str(product))


def display_summary_tables(tables):
    integer_cols = {"n_obs", "n_stations", "n_station_series", "n_station_spearman"}
    abs_rows = []
    for row in tables.get("metrics_absolute_overall", []):
        abs_rows.append({
            "method": product_label(row.get("product")),
            "n_obs": row.get("n_obs"),
            "n_stations": row.get("n_stations"),
            "bias": row.get("bias"),
            "mae": row.get("mae"),
            "rmse": row.get("rmse"),
            "spearman": row.get("spearman_r"),
        })
    _html_table("Method-to-reference absolute performance", _round_table(abs_rows, integer_cols), ["method", "n_obs", "n_stations", "bias", "mae", "rmse", "spearman"])

    orbital_rows = []
    for row in tables.get("metrics_absolute_by_orbital_phase", []):
        orbital_rows.append({
            "product": product_label(row.get("product")),
            "orbital_phase": row.get("orbital_phase"),
            "n_obs": row.get("n_obs"),
            "n_stations": row.get("n_stations"),
            "bias": row.get("bias"),
            "mae": row.get("mae"),
            "rmse": row.get("rmse"),
            "spearman": row.get("spearman_r"),
        })
    _html_table("SWOT orbital-phase performance", _round_table(orbital_rows, integer_cols), ["product", "orbital_phase", "n_obs", "n_stations", "bias", "mae", "rmse", "spearman"])

    hydro_rows = []
    for row in tables.get("metrics_absolute_by_hydro_phase", []):
        hydro_rows.append({
            "product": product_label(row.get("product")),
            "hydro_phase": row.get("hydro_phase"),
            "n_obs": row.get("n_obs"),
            "bias": row.get("bias"),
            "mae": row.get("mae"),
            "rmse": row.get("rmse"),
            "spearman": row.get("spearman_r"),
        })
    _html_table("SWOT hydrological-phase performance", _round_table(hydro_rows, integer_cols), ["product", "hydro_phase", "n_obs", "bias", "mae", "rmse", "spearman"])

    dynamic_rows = []
    for row in tables.get("metrics_dynamic_overall", []):
        dynamic_rows.append({
            "method": product_label(row.get("product")),
            "n_stations": row.get("n_stations"),
            "n_obs": row.get("n_obs"),
            "spearman_r_mean": row.get("spearman_r_mean"),
            "spearman_r_median": row.get("spearman_r_median"),
        })
    _html_table("Dynamic validation summary", _round_table(dynamic_rows, integer_cols), ["method", "n_stations", "n_obs", "spearman_r_mean", "spearman_r_median"])

    station_rows = []
    for row in tables.get("metrics_absolute_by_station", []):
        station_rows.append({
            "method": product_label(row.get("product")),
            "station": row.get("station"),
            "n_obs": row.get("n_obs"),
            "bias": row.get("bias"),
            "mae": row.get("mae"),
            "rmse": row.get("rmse"),
            "spearman": row.get("spearman_r"),
        })
    station_rows = sorted(station_rows, key=lambda r: (str(r.get("method", "")), str(r.get("station", ""))))
    _html_table("Station-level absolute performance", _round_table(station_rows, integer_cols), ["method", "station", "n_obs", "bias", "mae", "rmse", "spearman"], max_rows=20)


def build_empty_reason(summary, load_audit):
    reasons = defaultdict(int)
    for row in load_audit:
        reasons[row.get("reason", "unknown")] += 1
    return {
        "summary": summary,
        "load_reasons": dict(sorted(reasons.items())),
        "hint": "Check data_load_audit.csv and outlier_filter_audit.csv for the station/product step that became empty.",
    }


def run_pipeline():
    ensure_dirs()
    metadata = load_metadata()
    stages, audit = build_matchups(metadata)
    absolute_stages = split_by_station_group(stages, "leveled")
    temporal_stages = {stage: list(rows) for stage, rows in stages.items()}

    out_abs = cfg_path("absolute_dir")
    out_tmp = cfg_path("temporal_dir")
    out_drv = cfg_path("drivers_dir")

    tables = {f"matchups_{stage}": rows for stage, rows in stages.items()}
    write_stage_tables(out_abs, absolute_stages)
    write_stage_tables(out_tmp, temporal_stages)
    write_table(out_abs / "outlier_filter_audit.csv", [r for r in audit if r.get("station_group") == "leveled"], OUTLIER_AUDIT_FIELDS)
    write_table(out_tmp / "outlier_filter_audit.csv", audit, OUTLIER_AUDIT_FIELDS)
    write_table(out_abs / "data_load_audit.csv", DATA_LOAD_AUDIT, LOAD_AUDIT_FIELDS)
    write_table(out_tmp / "data_load_audit.csv", DATA_LOAD_AUDIT, LOAD_AUDIT_FIELDS)

    absolute = absolute_stages["outlier_filtered"]
    temporal = temporal_stages["outlier_filtered"]
    tables["metrics_absolute_overall"] = aggregate_metrics(absolute, ["product"])
    tables["metrics_absolute_by_station"] = aggregate_metrics(absolute, ["station", "product"])
    tables["metrics_absolute_by_orbital_phase"] = aggregate_metrics(absolute, ["product", "orbital_phase"])
    tables["metrics_absolute_by_hydro_phase"] = aggregate_metrics([r for r in absolute if r.get("product") in PRODUCTS_SWOT], ["product", "hydro_phase"])
    tables["metrics_dynamic_by_station"], tables["metrics_dynamic_overall"], tables["normalized_series_for_examples"] = dynamic_metrics(temporal)
    tables["driver_matrix"], tables["driver_spearman"] = driver_tables(tables["metrics_absolute_by_station"], metadata)

    absolute_outputs = {
        "metrics_absolute_overall": ["metrics_absolute_overall.csv", "metrics_overall.csv"],
        "metrics_absolute_by_station": ["metrics_absolute_by_station.csv", "metrics_by_station.csv"],
        "metrics_absolute_by_orbital_phase": ["metrics_absolute_by_orbital_phase.csv", "metrics_by_product_orbital_phase.csv"],
        "metrics_absolute_by_hydro_phase": ["metrics_absolute_by_hydro_phase.csv", "error_by_hydrological_phase.csv"],
    }
    temporal_outputs = {
        "metrics_dynamic_by_station": ["metrics_dynamic_by_station.csv"],
        "metrics_dynamic_overall": ["metrics_dynamic_overall.csv"],
        "normalized_series_for_examples": ["normalized_series_for_examples.csv"],
    }
    write_named_tables(out_abs, tables, absolute_outputs)
    write_named_tables(out_tmp, tables, temporal_outputs)
    write_table(out_drv / "station_level_error_predictor_matrix.csv", tables["driver_matrix"])
    write_table(out_drv / "station_level_spearman_summary.csv", tables["driver_spearman"], ["product", "variable", "spearman_rho", "n_stations"])

    summary = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "base_path": str(CFG["base_path"]),
        "date_start": CFG["date_start"],
        "date_end": CFG["date_end"],
        "match_window_days": CFG["match_window_days"],
        "n_metadata_stations": len(metadata),
        "n_load_audit_rows": len(DATA_LOAD_AUDIT),
        "n_matchups_raw_all": len(stages["raw"]),
        "n_matchups_quality_filtered_all": len(stages["quality_filtered"]),
        "n_matchups_outlier_filtered_all": len(stages["outlier_filtered"]),
        "n_matchups_absolute_outlier_filtered": len(absolute),
        "n_matchups_temporal_outlier_filtered": len(temporal),
        "n_absolute_station_product_metrics": len(tables["metrics_absolute_by_station"]),
        "n_dynamic_station_product_series": len(tables["metrics_dynamic_by_station"]),
    }
    diagnostics = build_empty_reason(summary, DATA_LOAD_AUDIT)
    (out_abs / "execution_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (out_tmp / "execution_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (out_abs / "pipeline_diagnostics.json").write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")
    (out_tmp / "pipeline_diagnostics.json").write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")
    tables["execution_summary"] = [summary]
    tables["figure_manifest"] = make_figures(tables)
    write_table(cfg_path("figures_main_dir") / "figure_manifest.csv", tables["figure_manifest"], ["figure_path"])

    if CFG.get("stop_if_no_matchups", True):
        problems = []
        if not absolute:
            problems.append("absolute validation has zero outlier-filtered matchups")
        if not temporal:
            problems.append("dynamic validation has zero outlier-filtered matchups")
        if problems:
            raise RuntimeError("; ".join(problems) + f". Diagnostics were written to {out_abs / 'pipeline_diagnostics.json'} and {out_abs / 'data_load_audit.csv'}")
    return tables


In [ ]:
input_items = {
    "base_path": Path(CFG["base_path"]),
    "metadata_primary": pjoin(CFG["metadata_candidates"][0]),
    "metadata_secondary": pjoin(CFG["metadata_candidates"][1]),
    "in_situ_unleveled": cfg_path("in_situ_unleveled_dir"),
    "in_situ_gnss_leveled": cfg_path("in_situ_gnss_leveled_dir"),
    "station_product_root": pjoin(Path(CFG["station_dir_template"].split("/Estacao_")[0])),
}

for name, path in input_items.items():
    print(f"{name}: {path} | exists={path.exists()}")

metadata_preview = load_metadata()
print(f"stations detected: {len(metadata_preview)}")
print(f"leveled stations detected: {len(available_leveled_stations())}")
print(f"station folders detected: {len(station_dirs())}")


In [ ]:
tables = run_pipeline()
display_summary_tables(tables)


In [ ]:
summary_path = cfg_path("absolute_dir") / "execution_summary.json"
print(summary_path)
print(summary_path.read_text(encoding="utf-8"))


## Main Outputs

Absolute validation tables are written to `analysis_absolute/`:

- `matchups_raw.csv`
- `matchups_quality_filtered.csv`
- `matchups_outlier_filtered.csv`
- `metrics_overall.csv`
- `metrics_by_station.csv`
- `metrics_by_product_orbital_phase.csv`
- `error_by_hydrological_phase.csv`
- `outlier_filter_audit.csv`
- `data_load_audit.csv`
- `pipeline_diagnostics.json`
- `execution_summary.json`

Dynamic validation tables, using all station matchups, are written to `analysis_temporal/`:

- `matchups_raw.csv`
- `matchups_quality_filtered.csv`
- `matchups_outlier_filtered.csv`
- `metrics_dynamic_by_station.csv`
- `metrics_dynamic_overall.csv`
- `normalized_series_for_examples.csv`
- `outlier_filter_audit.csv`
- `data_load_audit.csv`
- `pipeline_diagnostics.json`
- `execution_summary.json`

Driver-analysis support tables are written to `analysis_drivers/`. Figures are written to `figures/main/` when `matplotlib` is available.
